# CMOR example with prototype CMIP6 tables/CVs

This code is a demonstration of the use of CMOR with the MIP tables in this repository and prototype CVs. This requires a conda environment with CMOR v3.15.1 installed (see cmor_environment.yml in this repository)

This example is up to date with the CMIP6 global attributes, including file and directory naming.

In [71]:
%time
import cmor
import numpy 
import json

CPU times: user 2 μs, sys: 1e+03 ns, total: 3 μs
Wall time: 4.53 μs


Identify CMOR version

In [72]:
{i:j for i,j in vars(cmor).items() if 'CMOR_VERSION' in i}

{'CMOR_VERSION_MAJOR': 3, 'CMOR_VERSION_MINOR': 15, 'CMOR_VERSION_PATCH': 1}

The following is adapted from the code at https://github.com/PCMDI/cmor/blob/main/Test/test_cmor_CMIP7.py

Construct the input json file for CMOR


In [111]:
%time

DATASET_INFO = {
    "_AXIS_ENTRY_FILE": "tables/UKCI_coordinate.json",
    "_FORMULA_VAR_FILE": "tables/UKCI_formula_terms.json",
    "_cmip6_option": 1,
    "_controlled_vocabulary_file": "tables/UKCI_CV.json",  # SEPARATE TO MIP TABLES FOR TESTING ONLY
    "activity_id": "CMIP",
    "calendar": "360_day",
    "forcing_index": "1",
    "grid_label": "gm",
    "initialization_index": "1",
    "institution_id": "MOHC",
    "nominal_resolution": "100 km",
    "physics_index": "1",
    "realization_index": "1",
    "source_id": "HadGEM3-GC31-LL",
    "frequency": "mon",
    "mip_era": "UKCI",
    "Conventions": "CF-1.12",
    "grid": "gm",
    "license": "UKCI data produced by MOHC is licensed under the Open Government License v3 (https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/)",
    "outpath": ".",
    "experiment_id": "1pctCO2",
    "source_type": "AOGCM",
    "parent_activity_id": "none",
    "parent_experiment_id": "none",
    "sub_experiment_id": "none",
    "table_id": "Amon",
    "required_model_components": "AOGCM",
    "branch_time_in_child": "0.0D0",
    "branch_time_in_parent": "0.0D0",
    "parent_time_units": "days since 1850-01-01 00:00:00",
    "parent_variant_label": "r1i1p1f1",
    "parent_source_id": "HadGEM3-GC31-LL",
    "parent_mip_era": "CMIP6",
    "branch_method": "no parent",
    "sub_experiment": "none"
}

with open('input.json', 'w') as fh:
    json.dump(DATASET_INFO, fh, indent=2)

CPU times: user 1e+03 ns, sys: 0 ns, total: 1e+03 ns
Wall time: 7.15 μs


In [112]:
%time
cmor.setup(inpath="tables", netcdf_file_action=cmor.CMOR_REPLACE)

cmor.dataset_json('input.json')

tos = numpy.array([27.] * 24)
tos = tos.reshape(2, 3, 4)

lat = numpy.array([10., 20., 30.])
lat_bnds = numpy.array([5., 15., 25., 35.])
lon = numpy.array([0., 90., 180., 270.])
lon_bnds = numpy.array([-45., 45.,
                        135.,
                        225.,
                        315.
                        ])
time = numpy.array([15.0, 45.0])
time_bnds = numpy.array([0.0, 30.0, 60.0])

CPU times: user 1 μs, sys: 0 ns, total: 1 μs
Wall time: 3.81 μs


In [113]:
%time
realm = "Amon"
cmor.load_table(f"UKCI_{realm}.json")

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 3.81 μs


0

In [114]:
%time
cmorlat = cmor.axis("latitude",
                    coord_vals=lat,
                    cell_bounds=lat_bnds,
                    units="degrees_north")
cmorlon = cmor.axis("longitude",
                    coord_vals=lon,
                    cell_bounds=lon_bnds,
                    units="degrees_east")
cmortime = cmor.axis("time",
                    coord_vals=time,
                    cell_bounds=time_bnds,
                    units="days since 2018")
axes = [cmortime, cmorlat, cmorlon]

CPU times: user 1 μs, sys: 1 μs, total: 2 μs
Wall time: 3.58 μs


In [115]:
variable = "hfss"
cmortos = cmor.variable(variable, "W m-2", axes, positive = "up")


In [116]:
cmor.write(cmortos, tos)
filename = cmor.close(cmortos, file_name=True)
filename

'./UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260915/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc'

Locate generated file

In [117]:
!find UKCI -type f -name '*.nc'

UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260915/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc


Dump header of generated file

In [118]:
!ncdump -h UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260915/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc

netcdf hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802 {
dimensions:
	time = UNLIMITED ; // (2 currently)
	lat = 3 ;
	lon = 4 ;
	bnds = 2 ;
variables:
	double time(time) ;
		time:bounds = "time_bnds" ;
		time:units = "days since 2018" ;
		time:calendar = "360_day" ;
		time:axis = "T" ;
		time:long_name = "time" ;
		time:standard_name = "time" ;
	double time_bnds(time, bnds) ;
	double lat(lat) ;
		lat:bounds = "lat_bnds" ;
		lat:units = "degrees_north" ;
		lat:axis = "Y" ;
		lat:long_name = "Latitude" ;
		lat:standard_name = "latitude" ;
	double lat_bnds(lat, bnds) ;
	double lon(lon) ;
		lon:bounds = "lon_bnds" ;
		lon:units = "degrees_east" ;
		lon:axis = "X" ;
		lon:long_name = "Longitude" ;
		lon:standard_name = "longitude" ;
	double lon_bnds(lon, bnds) ;
	float hfss(time, lat, lon) ;
		hfss:standard_name = "surface_upward_sensible_heat_flux" ;
		hfss:long_name = "Surface Upward Sensible Heat Flux" ;
		hfss:comment = "The surface sensible heat flux, also called turbulent 

In [ ]:
# # clean up
#! rm -rf MIP-DRS7
#! rm -rf input.json